# Actualizar APRX Imagenes Drone

Agrega las imagenes cargadas al grupo correcto, renombra las capas a formato corto y ordena de mas nuevas a mas viejas usando la fecha del nombre.


In [1]:
from pathlib import Path
import csv
import re
import arcpy

# Entradas principales
LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"

APRX_PATH = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
MAP_NAME = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
PARENT_GROUP_NAME = "Vuelos Drone PAO"
TARGET_GROUP_NAME = "Imagenes Drone"

PREFIX = "CL_MLP_PAO_IF_Ortho_"
SUFFIX = ".tif"

SAVE_COPY_FOR_REVIEW = True
APRX_COPY_PATH = str(Path.cwd() / "APRX" / f"{Path(APRX_PATH).stem}_verificacion_imagenes_drone.aprx")
ADD_ONLY_MISSING_TO_GROUP = True


def read_loaded_image_paths(csv_path):
    with Path(csv_path).open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [
            row["destination_path"]
            for row in rows
            if row.get("overall_status") == "ok" and row.get("destination_path")
        ]


def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def short_image_name(value):
    name = Path(str(value)).name
    while name.startswith("tmp_"):
        name = name.replace("tmp_", "", 1)
    if name.startswith(PREFIX):
        name = name.replace(PREFIX, "", 1)
    if name.endswith(SUFFIX):
        name = name[:-len(SUFFIX)]
    return name


def image_keys(value):
    short = short_image_name(value)
    return {
        short.lower(),
        f"{PREFIX}{short}".lower(),
        f"{PREFIX}{short}{SUFFIX}".lower(),
        f"{short}{SUFFIX}".lower(),
    }


def image_date_sort_key(name):
    short = short_image_name(name)
    match = re.match(r"^(\d{2})_(\d{2})_(\d{2})_(.+)$", short)
    if not match:
        return (0, 0, 0, short.lower())
    yy, mm, dd, rest = match.groups()
    return (int(yy), int(mm), int(dd), rest.lower())


def find_group(map_obj, group_name, parent_group=None):
    groups = [layer for layer in map_obj.listLayers() if layer.isGroupLayer]
    matches = []
    for group in groups:
        if group.name != group_name:
            continue
        if parent_group is not None:
            expected_prefix = layer_long_name(parent_group) + "\\"
            if not layer_long_name(group).startswith(expected_prefix):
                continue
        matches.append(group)

    if not matches:
        available = [layer_long_name(layer) for layer in groups]
        parent_text = f" bajo {layer_long_name(parent_group)}" if parent_group else ""
        raise ValueError(f"No se encontro grupo '{group_name}'{parent_text}. Grupos disponibles: {available}")

    if len(matches) > 1:
        print(f"Advertencia: se encontraron {len(matches)} grupos '{group_name}'. Se usara {layer_long_name(matches[0])}")

    return matches[0]


def is_direct_child(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    prefix = group_long_name + "\\"
    if not long_name.startswith(prefix):
        return False
    relative = long_name[len(prefix):]
    return "\\" not in relative


def direct_raster_children(map_obj, group_layer):
    return [
        layer
        for layer in map_obj.listLayers()
        if layer != group_layer and is_direct_child(layer, group_layer) and layer.isRasterLayer and not layer.isGroupLayer
    ]


def add_raster_directly_to_group(map_obj, group_layer, raster_path, target_name):
    before = {layer_long_name(layer) for layer in direct_raster_children(map_obj, group_layer)}
    temp_layer = map_obj.addDataFromPath(raster_path)
    temp_layer.name = target_name

    try:
        map_obj.addLayerToGroup(group_layer, temp_layer, "TOP")
    finally:
        map_obj.removeLayer(temp_layer)

    after_layers = direct_raster_children(map_obj, group_layer)
    new_layers = [layer for layer in after_layers if layer_long_name(layer) not in before]
    if new_layers:
        new_layers[0].name = target_name
        return new_layers[0]

    # ArcPy a veces no permite identificar el retorno; buscar por nombres equivalentes.
    target_keys = image_keys(target_name)
    for layer in after_layers:
        if image_keys(layer.name).intersection(target_keys):
            layer.name = target_name
            return layer

    return None


def move_to_top_inside_group(map_obj, group_layer, layer_to_move):
    children = direct_raster_children(map_obj, group_layer)
    if not children or children[0] == layer_to_move:
        return
    map_obj.moveLayer(children[0], layer_to_move, "BEFORE")


def order_group_newest_first(map_obj, group_layer):
    ordered = sorted(
        direct_raster_children(map_obj, group_layer),
        key=lambda layer: image_date_sort_key(layer.name),
        reverse=True,
    )
    # Insertar de atras hacia adelante para dejar arriba la mas nueva.
    for layer in reversed(ordered):
        move_to_top_inside_group(map_obj, group_layer, layer)


def normalize_group_layer_names(map_obj, group_layer):
    renamed = 0
    for layer in direct_raster_children(map_obj, group_layer):
        new_name = short_image_name(layer.name)
        if layer.name != new_name:
            print(f"Renombrando: {layer.name} -> {new_name}")
            layer.name = new_name
            renamed += 1
    return renamed


In [2]:
image_paths_to_add = read_loaded_image_paths(LOAD_RESULTS_CSV)
print(f"Imagenes del CSV para revisar en APRX: {len(image_paths_to_add)}")
print(f"APRX: {APRX_PATH}")
print(f"Mapa: {MAP_NAME}")
print(f"Grupo destino: {PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}")

aprx = arcpy.mp.ArcGISProject(APRX_PATH)

try:
    maps = aprx.listMaps(MAP_NAME)
    if not maps:
        raise ValueError(f"No se encontro el mapa: {MAP_NAME}")

    map_obj = maps[0]
    parent_group = find_group(map_obj, PARENT_GROUP_NAME)
    target_group = find_group(map_obj, TARGET_GROUP_NAME, parent_group)

    print(f"Grupo padre encontrado: {layer_long_name(parent_group)}")
    print(f"Grupo destino encontrado: {layer_long_name(target_group)}")

    existing_keys = set()
    for layer in direct_raster_children(map_obj, target_group):
        existing_keys.update(image_keys(layer.name))

    added_count = 0
    skipped_count = 0
    error_count = 0

    for raster_path in image_paths_to_add:
        target_name = short_image_name(raster_path)
        target_keys = image_keys(target_name)

        if ADD_ONLY_MISSING_TO_GROUP and target_keys.intersection(existing_keys):
            print(f"Ya existe, se omite: {target_name}")
            skipped_count += 1
            continue

        try:
            layer = add_raster_directly_to_group(map_obj, target_group, raster_path, target_name)
            if layer is None:
                print(f"Advertencia: capa agregada no confirmada por ArcPy: {target_name}")
            existing_keys.update(target_keys)
            added_count += 1
            print(f"Agregada: {target_name}")
        except Exception as exc:
            error_count += 1
            print(f"ERROR agregando {raster_path}: {exc}")

    renamed_count = normalize_group_layer_names(map_obj, target_group)
    order_group_newest_first(map_obj, target_group)

    final_layers = direct_raster_children(map_obj, target_group)
    print("Orden final, primeras 15 capas:")
    for layer in final_layers[:15]:
        print(f"- {layer.name}")

    print(
        f"Resumen APRX: agregadas={added_count}, omitidas_existentes={skipped_count}, "
        f"renombradas={renamed_count}, errores={error_count}, total_grupo={len(final_layers)}"
    )

    if SAVE_COPY_FOR_REVIEW:
        Path(APRX_COPY_PATH).parent.mkdir(parents=True, exist_ok=True)
        aprx.saveACopy(APRX_COPY_PATH)
        print(f"Copia de revision guardada: {APRX_COPY_PATH}")
    else:
        aprx.save()
        print("APRX original guardado")
finally:
    del aprx


Imagenes del CSV para revisar en APRX: 32
APRX: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx
Mapa: CL MLP PAO 27 Imagenes Aereas PAO Image Server
Grupo destino: Vuelos Drone PAO > Imagenes Drone
Grupo padre encontrado: Vuelos Drone PAO
Grupo destino encontrado: Vuelos Drone PAO\Imagenes Drone
Agregada: 26_05_01_Estacion_Cabecera
Agregada: 26_05_06_Subestacion-El-Mauro-1
Agregada: 26_05_06_Subestacion-El-Mauro-2
Agregada: 26_05_06_TORRE_E85_A_E_125
Agregada: 26_05_10_ED1
Agregada: 26_05_10_Helipuerto
Agregada: 26_05_10_Patio-19B-y-Armado
Agregada: 26_05_10_DME9-PA12-IIFF8
Agregada: 26_05_07_TORRES_E48_A_E84_PV4
Agregada: 26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3
Agregada: 26_05_13_DME9-PA12-IIFF8
Agregada: 26_05_13_Subestacion-El-Mauro_A_E35
Agregada: 26_05_13_Subestacion-El-Mauro-1
Agregada: 26_05_13_EM2_S2
Agregada: 26_05_13_EBD-1
Agregada: 26_05_13_ED2
Agregada: 26_05_13_Subestacion-El-Mauro-2
Agregada: 26_05_13_EBD-2
Agregada: 26_05

In [25]:
import arcpy
import pandas as pd
import re 
from arcgis import GIS
import numpy as np
import os

In [45]:
fc_footprint = r'\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'

bkg = rf'{Path.cwd()}\Dataset\Datos.gdb'
fc_footprint = rf'{bkg}\{Path(fc_footprint).name}'
sdf = pd.DataFrame.spatial.from_featureclass(fc_footprint)
del sdf['index']
colsr = {'name': 'Name',
 'min_ps': 'MinPS',
 'max_ps': 'MaxPS',
 'low_ps': 'LowPS',
 'high_ps': 'HighPS',
 'category': 'Category',
 'tag': 'Tag',
 'group_name': 'GroupName',
 'product_nam': 'ProductNam',
 'center_x': 'CenterX',
 'center_y': 'CenterY',
 'z_order': 'ZOrder',
 'type_id': 'TypeID',
 'item_ts': 'ItemTS',
 'uri_hash': 'UriHash',
 'sector': 'Sector',
 'sensor': 'Sensor',
 'proyecto': 'Proyecto',
 'fecha_adqu': 'Fecha_Adqu',
 'fecha_publ': 'Fecha_Publ',
 'estado': 'Estado',
 'url': 'URL',
 'url2': 'URL2',
 'nombre_de_vuelo': 'Nombre_de_Vuelo'}
sdf = sdf.replace({np.nan:None})
sdf = sdf.rename(columns=colsr)
sdf = sdf.sort_values(['Fecha_Adqu','Name'],ascending=[False,True]).reset_index(drop=True)
del sdf['OBJECTID']

print(f'total de registros {sdf.shape[0]}')
sdf.head()

total de registros 481


,Name,MinPS,MaxPS,LowPS,HighPS,Category,Tag,GroupName,ProductNam,CenterX,...,Sector,Sensor,Proyecto,Fecha_Adqu,Fecha_Publ,Estado,URL,URL2,Nombre_de_Vuelo,SHAPE
0,CL_MLP_PAO_IF_Ortho_26_05_17_EDT,0.0,10000.0,0.15,0.053135,1,Dataset,,20855,263894.998652,...,EDT,DJI Mavic Enterprise,PAO,2026-05-17,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,480_26_05_17_EDT,"{""rings"": [[[-7958918.7502, -3747990.228700001..."
1,CL_MLP_PAO_IF_Ortho_26_05_17_EV2,0.0,10000.0,0.15,0.091167,1,Dataset,,20854,295994.841402,...,EV2,DJI Mavic Enterprise,PAO,2026-05-17,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,481_26_05_17_EV2,"{""rings"": [[[-7921381.1171, -3755775.557599999..."
2,CL_MLP_PAO_IF_Ortho_26_05_16_TORRES_E48_A_E84_PV4,0.0,10000.0,0.15,0.871391,1,Dataset,,20853,302191.717517,...,TORRES_E48_A_E84_PV4,DJI Mavic Enterprise,PAO,2026-05-16,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,479_26_05_16_TORRES_E48_A_E84_PV4,"{""rings"": [[[-7912819.7358, -3758095.5933], [-..."
3,CL_MLP_PAO_IF_Ortho_26_05_15_EM1,0.0,10000.0,0.15,0.161235,1,Dataset,,20857,347130.874954,...,EM1,DJI Mavic Enterprise,PAO,2026-05-15,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,474_26_05_15_EM1,"{""rings"": [[[-7860973.0942, -3743155.956799999..."
4,CL_MLP_PAO_IF_Ortho_26_05_15_EV1,0.0,10000.0,0.15,0.174002,1,Dataset,,20851,345367.551764,...,EV1,DJI Mavic Enterprise,PAO,2026-05-15,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,476_26_05_15_EV1,"{""rings"": [[[-7863269.934900001, -3748385.9510..."


In [47]:

f = r'\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'

# c = [c.name for c in arcpy.ListFields(f) if not re.search(r'(shape)|(objectid)',c.name, re.IGNORECASE)]
# c1 =[c for c in sdf.columns if not re.search(r'(shape)|(objectid)',c, re.IGNORECASE)]
# dict(zip(c1,c))

In [ ]:
# bkg = rf'{Path.cwd()}\Dataset\Datos.gdb'
# if not os.path.exists(bkg):
#     arcpy.CreateFileGDB_management(f'{Path(bkg).parent}',f'{Path(bkg).name}')
# sdf.spatial.to_featureclass(rf'{bkg}\{Path(fc_footprint).name}')

'c:\\Users\\esrlrivero_adm\\Documents\\Geosupport\\amsa-pao-geosupport\\Dataset\\Datos.gdb\\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'

In [ ]:
# ## 
# arcpy.TruncateTable_management(f)
# cols_sdf = [c for c in sdf.columns if not re.search('shape',c, re.IGNORECASE)]
# cols = cols_sdf + ['SHAPE@']
# with arcpy.da.InsertCursor(f,cols) as cursor:
#     for row in sdf.itertuples(index=False):
#         valores = [getattr(row,campo) for campo in cols_sdf]
#         geom = getattr(row,'SHAPE')
#         geom = geom.as_arcpy
#         valores.append(geom)
#         cursor.insertRow(valores)        

    

In [51]:
row

Pandas(Name='CL_MLP_PAO_IF_Ortho_25_01_08_EB1', MinPS=0.0, MaxPS=0.186666667, LowPS=0.0115, HighPS=0.092, Category=1, Tag='Dataset', GroupName='', ProductNam='10623', CenterX=263861.422301918, CenterY=6469668.469567813, ZOrder=None, TypeID=21, ItemTS=46125.65629662037, UriHash='0C075F66D8302F7491F15941E056C304', Sector='EB1', Sensor='DJI MATRICE 350 RTK', Proyecto='PAO', Fecha_Adqu=Timestamp('2025-01-08 04:00:00'), Fecha_Publ=Timestamp('2025-07-30 00:00:00'), Estado='Activo', URL='https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/CL_MLP_PAO_IF_Ortho_ALL/ImageServer/file?id=.\\Puerto_Punta_Chungo_Drone\\25_01_08\\CL_MLP_PAO_IF_Ortho_25_01_08_EB1.tiff&rasterId=', URL2='https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/CL_MLP_PAO_IF_Ortho_ALL/ImageServer/file?id=.\\Puerto_Punta_Chungo_Drone\\25_01_08\\CL_MLP_PAO_IF_Ortho_25_01_08_EB1.tiff&rasterId=', Nombre_de_Vuelo='001_25_01_08_EB1', SHAPE={'rings': [[[-7958808.441, -3747973.4750000015], [-7958806.6735, -3747984.3156999983

In [50]:
df = pd.DataFrame.spatial.from_featureclass(f)
df.head()

,OBJECTID,Name,MinPS,MaxPS,LowPS,HighPS,Category,Tag,GroupName,ProductNam,...,Sector,Sensor,Proyecto,Fecha_Adqu,Fecha_Publ,Estado,URL,URL2,Nombre_de_Vuelo,SHAPE
0,1,CL_MLP_PAO_IF_Ortho_26_05_17_EDT,0.0,10000.0,0.15,0.053135,1,Dataset,,20855,...,EDT,DJI Mavic Enterprise,PAO,2026-05-17,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,480_26_05_17_EDT,"{""rings"": [[[-7958918.7502, -3747990.228700001..."
1,2,CL_MLP_PAO_IF_Ortho_26_05_17_EV2,0.0,10000.0,0.15,0.091167,1,Dataset,,20854,...,EV2,DJI Mavic Enterprise,PAO,2026-05-17,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,481_26_05_17_EV2,"{""rings"": [[[-7921381.1171, -3755775.557599999..."
2,3,CL_MLP_PAO_IF_Ortho_26_05_16_TORRES_E48_A_E84_PV4,0.0,10000.0,0.15,0.871391,1,Dataset,,20853,...,TORRES_E48_A_E84_PV4,DJI Mavic Enterprise,PAO,2026-05-16,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,479_26_05_16_TORRES_E48_A_E84_PV4,"{""rings"": [[[-7912819.7358, -3758095.5933], [-..."
3,4,CL_MLP_PAO_IF_Ortho_26_05_15_EM1,0.0,10000.0,0.15,0.161235,1,Dataset,,20857,...,EM1,DJI Mavic Enterprise,PAO,2026-05-15,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,474_26_05_15_EM1,"{""rings"": [[[-7860973.0942, -3743155.956799999..."
4,5,CL_MLP_PAO_IF_Ortho_26_05_15_EV1,0.0,10000.0,0.15,0.174002,1,Dataset,,20851,...,EV1,DJI Mavic Enterprise,PAO,2026-05-15,2026-06-15,Activo,https://sig.aminerals.cl/imgdyn/rest/services/...,,476_26_05_15_EV1,"{""rings"": [[[-7863269.934900001, -3748385.9510..."
